# Model Training & Evaluation

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("cleaned_ratings.csv")
print(f"Loaded: {len(df):,} ratings | "
      f"{df['book_id'].nunique():,} books | "
      f"{df['user_id'].nunique():,} users")

Loaded: 4,044,839 ratings | 22,931 books | 61,078 users


## 1. Train/Validation/Test Split

In [2]:
# Standard 80/10/10 train/val/test split
n = len(df)
test_size = int(n * 0.10)
val_size  = int(n * 0.10)

test  = df.sample(n=test_size, random_state=42)
rest  = df.drop(test.index)
val   = rest.sample(n=val_size, random_state=42)
train = rest.drop(val.index)

train = train.reset_index(drop=True)
val   = val.reset_index(drop=True)
test  = test.reset_index(drop=True)

print(f"Train: {len(train):,}  ({len(train)/n*100:.1f}%)")
print(f"Val:   {len(val):,}   ({len(val)/n*100:.1f}%)")
print(f"Test:  {len(test):,}   ({len(test)/n*100:.1f}%)")

# Cold-start check
for name, split in [("Val", val), ("Test", test)]:
    cold_u = set(split["user_id"]) - set(train["user_id"])
    cold_b = set(split["book_id"]) - set(train["book_id"])
    print(f"{name} cold-start — users: {len(cold_u)}, books: {len(cold_b)}")

train.to_csv("train_ratings.csv", index=False)
val.to_csv("val_ratings.csv",     index=False)
test.to_csv("test_ratings.csv",   index=False)
print("\nSaved: train_ratings.csv, val_ratings.csv, test_ratings.csv")

Train: 3,235,873  (80.0%)
Val:   404,483   (10.0%)
Test:  404,483   (10.0%)
Val cold-start — users: 0, books: 0
Test cold-start — users: 0, books: 0

Saved: train_ratings.csv, val_ratings.csv, test_ratings.csv


## 2. Train/Val/Test Validation

In [3]:
train = pd.read_csv("train_ratings.csv")
val   = pd.read_csv("val_ratings.csv")
test  = pd.read_csv("test_ratings.csv")

ORIGINAL_TOTAL = 4_044_839
n = len(train) + len(val) + len(test)

print(f"{'CHECK':<45} RESULT")
print("─" * 68)

# 1. Total count preserved
status = "✓" if n == ORIGINAL_TOTAL else f"⚠ expected {ORIGINAL_TOTAL:,}"
print(f"{'1. Total count':<45} {n:,}  {status}")

# 2. Proportions (~80 / 10 / 10)
print(f"{'2. Proportions (train/val/test)':<45} "
      f"{len(train)/n*100:.1f}% / {len(val)/n*100:.1f}% / {len(test)/n*100:.1f}%")

# 3. No nulls
for name, df in [("Train", train), ("Val", val), ("Test", test)]:
    nulls = df.isnull().sum().sum()
    print(f"{'3. Nulls (' + name + ')':<45} {'✓ None' if nulls == 0 else f'⚠ {nulls}'}")

# 4. Rating range [1–5]
for name, df in [("Train", train), ("Val", val), ("Test", test)]:
    rmin, rmax = df["rating"].min(), df["rating"].max()
    ok = rmin >= 1 and rmax <= 5
    print(f"{'4. Rating range (' + name + ')':<45} {'✓' if ok else '⚠'} [{rmin}, {rmax}]")

# 5. No cross-split overlap
# Index-based splitting + total count preservation guarantees disjoint splits
# (same (user_id, book_id) pair cannot appear in two splits since original has no duplicates)
print(f"{'5. No cross-split overlap':<45} "
      f"{'✓ guaranteed (index-based split + total preserved)' if n == ORIGINAL_TOTAL else '⚠'}")

# 6. Cold-start check
cold_val_u  = len(set(val["user_id"])  - set(train["user_id"]))
cold_test_u = len(set(test["user_id"]) - set(train["user_id"]))
cold_val_b  = len(set(val["book_id"])  - set(train["book_id"]))
cold_test_b = len(set(test["book_id"]) - set(train["book_id"]))
print(f"{'6a. Cold-start users (val / test)':<45} {cold_val_u} / {cold_test_u}")
print(f"{'6b. Cold-start books (val / test)':<45} {cold_val_b} / {cold_test_b}")

# 7. Rating distribution (should be similar across splits)
print(f"\n{'7. Mean rating':<45} "
      f"Train: {train['rating'].mean():.3f} | "
      f"Val: {val['rating'].mean():.3f} | "
      f"Test: {test['rating'].mean():.3f}")
print(f"{'   Std rating':<45} "
      f"Train: {train['rating'].std():.3f}  | "
      f"Val: {val['rating'].std():.3f}  | "
      f"Test: {test['rating'].std():.3f}")

CHECK                                         RESULT
────────────────────────────────────────────────────────────────────
1. Total count                                4,044,839  ✓
2. Proportions (train/val/test)               80.0% / 10.0% / 10.0%
3. Nulls (Train)                              ✓ None
3. Nulls (Val)                                ✓ None
3. Nulls (Test)                               ✓ None
4. Rating range (Train)                       ✓ [1, 5]
4. Rating range (Val)                         ✓ [1, 5]
4. Rating range (Test)                        ✓ [1, 5]
5. No cross-split overlap                     ✓ guaranteed (index-based split + total preserved)
6a. Cold-start users (val / test)             0 / 0
6b. Cold-start books (val / test)             0 / 0

7. Mean rating                                Train: 3.988 | Val: 3.986 | Test: 3.987
   Std rating                                 Train: 0.937  | Val: 0.937  | Test: 0.939
